# CADGenesis-LM v6.0 Colab Verification Notebook

This notebook verifies the production-upgrade work completed in the session,
including the new self-correction inference system and existing test suite.

**Hardware**: Google Colab (T4/A100 GPU available)
**VRAM**: 16GB (T4) or 40GB+ (A100 Pro+)
**Timeout**: Sessions auto-disconnect after ~90min (Free) or ~4-6hr (Pro)

---

**Run order matters**: Cell 1 must run before cells 2-5 install/load the package.

In [ ]:
# Cell 1: Setup & Installation
# Runs once at the top - installs the package in editable mode

import sys, os, subprocess, warnings
warnings.filterwarnings('ignore')

# Mount Google Drive (optional, for persistent storage)
from google.colab import drive
try:
    mount_point = '/content/drive'
    drive.mount(mount_point, force_remount=True)
    print('Google Drive mounted successfully')
except Exception as e:
    print(f'Drive mount skipped: {e}')

# Ensure project directory exists
proj_dir = '/content/cadgenesis-lm'  # or wherever the repo is
os.makedirs(proj_dir, exist_ok=True)

# Copy/symlink the source if needed
src_path = os.path.join(proj_dir, 'src')
if not os.path.exists(src_path):
    # Try to find the repo in common locations
    for candidate in ['/content/cadgenesis', '/content/CAD_LLM', '/content/cad_llm']:
        if os.path.exists(os.path.join(candidate, 'src')):
            src_path = candidate
            break

print(f'Using source path: {src_path}')

# Install cadgenesis in editable mode
print('\nInstalling cadgenesis...')
result = subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', src_path],
    capture_output=True, text=True
)
print('STDOUT:', result.stdout[-300:] if len(result.stdout) > 300 else result.stdout)
if result.stderr:
    print('STDERR:', result.stderr[-300:] if len(result.stderr) > 300 else result.stderr)
print(f'Install return code: {result.returncode}')

# Verify installation
result = subprocess.run(
    [sys.executable, '-c', 'import cadgenesis; print("cadgenesis version:", cadgenesis.__version__)'],
    capture_output=True, text=True
)
print(result.stdout)
if result.stderr:
    print('STDERR:', result.stderr)

print('\n=== Installation Complete ===')

In [ ]:
# Cell 2: Verify Self-Correction System

import sys
sys.path.insert(0, '/content/src')

# Now these should work after pip install
from cadgenesis.inference.self_correction import SelfCorrectingInference, SelfCorrectionResult
from cadgenesis.execution.geometry_validation import validate_program
from cadgenesis.confidence.risk import RiskAssessor, RiskConfig
from cadgenesis.confidence.monitoring import ConfidenceMonitor

print('=== Self-Correction System Tests ===')

# Test 1: Basic self-correction
inf = SelfCorrectingInference(max_attempts=3)
result = inf.correct('make a box 10x10', ['BOX', 'NUM_10', 'EXTRUDE', 'NUM_5'])
print(f'Test 1 - Valid program: success={result.success}, risk={result.risk_score}')

# Test 2: Program missing base operation
result2 = inf.correct('extrude shape', ['EXTRUDE', 'NUM_10'])
print(f'Test 2 - Missing base: success={result2.success}, error={result2.error[:60]}')

# Test 3: Program with EXTRUDE but no dimension
result3 = inf.correct('extrude a shape', ['EXTRUDE'])
print(f'Test 3 - Missing dimension: success={result3.success}, risk={result3.risk_score}')

# Test 4: Geometry validator
r = validate_program(['BOX', 'NUM_10', 'EXTRUDE', 'NUM_5'])
print(f'Test 4 - Geometry validator: valid={r}')

# Test 5: Risk assessor
ra = RiskAssessor(alpha=1.0, beta=1.0, gamma=1.0)
risk = ra.assess(confidence=0.9, uncertainty=0.1, consequence=0.5)
print(f'Test 5 - Risk assessor: risk_score={risk["risk_score"]:.4f}, action={risk["action"]}')

# Test 6: Confidence monitor
cm = ConfidenceMonitor()
cm.update([0.9, 0.8, 0.7, 0.6, 0.5])
summary = cm.summary()
print(f'Test 6 - ConfidenceMonitor: count={summary["count"]}, mean={summary["mean"]:.4f}')

print('\n=== All Self-Correction Tests Complete ===')

In [ ]:
# Cell 3: Run Subset of Existing Tests

import subprocess, sys

print('=== Running pytest Subset ===')

# Run tokenizer + dynamic routing tests
result = subprocess.run(
    [sys.executable, '-m', 'pytest',
     'tests/tokenizer/',
     'tests/transformer/test_dynamic_routing.py',
     '-q', '--tb=short'],
    capture_output=True, text=True, cwd='/content'
)

# Count passed/failed
passed = result.stdout.count('PASSED')
failed = result.stdout.count('FAILED')
errors = result.stdout.count('ERROR')

print(f'Tests passed: {passed}')
print(f'Tests failed: {failed}')
print(f'Test errors: {errors}')
print(f'Return code: {result.returncode}')

if result.returncode == 0:
    print('\n✓ All tests passed!')
else:
    print('\n⚠ Some tests had issues')
    print('First 500 chars of stderr:')
    print(result.stderr[:500] if result.stderr else 'None')

In [ ]:
# Cell 4: Ruff & Mypy Checks

import subprocess, sys

print('=== Linting & Type Checking ===')

# Ruff check on self_correction.py
result = subprocess.run(
    [sys.executable, '-m', 'ruff', 'check', 'src/cadgenesis/inference/self_correction.py'],
    capture_output=True, text=True, cwd='/content'
)
print(f'ruff check: {"PASS" if result.returncode == 0 else "FAIL"}')
if result.stdout:
    print(f'  Output: {result.stdout[:200]}')

# Ruff format check
result = subprocess.run(
    [sys.executable, '-m', 'ruff', 'format', '--check', 'src/cadgenesis/inference/self_correction.py'],
    capture_output=True, text=True, cwd='/content'
)
print(f'ruff format: {"PASS" if result.returncode == 0 else "FAIL"}')

# Mypy check
result = subprocess.run(
    [sys.executable, '-m', 'mypy', 'src/cadgenesis/inference/self_correction.py', '--ignore-missing-imports'],
    capture_output=True, text=True, cwd='/content'
)
inference_errs = [l for l in result.stdout.split('\n') if 'self_correction' in l]
print(f'mypy self_correction.py: {"PASS - 0 errors" if not inference_errs else f"{len(inference_errs)} errors"}')
if inference_errs:
    print(f'  Errors: {inference_errs[:3]}')

print('\n=== Linting/Type Checking Complete ===')

In [ ]:
# Cell 5: Quick Integration Verification

import sys
sys.path.insert(0, '/content/src')

print('=== Integration Verification ===')

# Verify all key imports work
try:
    from cadgenesis.inference.self_correction import SelfCorrectingInference, SelfCorrectionResult
    print('✓ self_correction import: OK')
except Exception as e:
    print(f'✗ self_correction import: FAIL - {e}')

try:
    from cadgenesis.confidence.risk import RiskAssessor, RiskConfig
    print('✓ risk import: OK')
except Exception as e:
    print(f'✗ risk import: FAIL - {e}')

try:
    from cadgenesis.confidence.monitoring import ConfidenceMonitor
    print('✓ monitoring import: OK')
except Exception as e:
    print(f'✗ monitoring import: FAIL - {e}')

try:
    from cadgenesis.execution.geometry_validation import validate_program
    print('✓ geometry_validator import: OK')
except Exception as e:
    print(f'✗ geometry_validator import: FAIL - {e}')

try:
    from cadgenesis.inference import SelfCorrectingInference
    print('✓ inference package import: OK')
except Exception as e:
    print(f'✗ inference package import: FAIL - {e}')

print('\n=== Integration Verification Complete ===')